# GARCH(1,1) Market Risk Model — ETERNAL.NS (Zomato)
### Dynamic Volatility, Value at Risk & Expected Shortfall
**Stock:** ETERNAL.NS | **Period:** July 2021 – May 2026 | **Model:** GARCH(1,1)

This notebook builds a dynamic market risk model from scratch using GARCH(1,1) conditional volatility estimation, 
99% Value at Risk forecasting, backtesting, and Expected Shortfall. 
Replicating the core framework used by risk desks under Basel III.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import scipy.stats as stats
from arch import arch_model

## 1. Data & Log Returns
Daily adjusted closing prices for ETERNAL.NS are sourced via yfinance from Zomato's NSE listing date. 
Log returns are computed as ln(Pt/Pt-1). The return series exhibits clear **volatility clustering** — 
large moves tend to follow large moves, and calm periods cluster together. This violates the constant 
variance assumption of simple models and motivates the use of GARCH.
    

In [ ]:
data = yf.download(tickers = "ETERNAL.NS", start = "2021-7-23", 
            auto_adjust = True)
price = data["Close"].dropna()

In [ ]:
price.tail()

In [ ]:
log_returns = np.log(price/ price.shift(1)).dropna().squeeze()
log_returns.describe()

In [ ]:
plt.figure(figsize = (12,8))
plt.plot(log_returns)
plt.xlabel("Dates", fontsize = 15)
plt.ylabel("Log Returns", fontsize = 15)
plt.title("ETERNAL.NS — Daily Log Returns (2021–2026)", fontsize = 20)
plt.show()

## 2. GARCH(1,1) Estimation
The GARCH(1,1) model estimates conditional variance as:

    σ²t = ω + α·ε²t-1 + β·σ²t-1

- **ω = 0.0000923** — variance intercept; implies long run daily volatility of ~3.02%
- **α = 0.0999** — shock sensitivity; large moves yesterday increase today's variance by ~10%
- **β = 0.7989** — persistence; 80% of yesterday's variance carries into today
- **α + β = 0.8988** — below 1, confirming the process is stationary and volatility mean-reverts

In [ ]:
model = arch_model(log_returns, p = 1, q = 1, vol = "Garch")
result = model.fit(disp = "off")

In [ ]:
result.summary()

In [ ]:
cond_vol = result.conditional_volatility.squeeze()
cond_vol

In [ ]:
plt.figure(figsize = (12,8))
plt.plot(cond_vol)
plt.xlabel("Dates", fontsize = 15)
plt.ylabel("Conditional Volatility", fontsize = 15)
plt.title("ETERNAL.NS — GARCH(1, 1) Conditional Volatility (2021–2026)", fontsize = 20)
plt.show()

## 3. Value at Risk (99%)
1-day 99% VaR is computed as:

    VaR = μ - z·σt    
    where z = 2.326 (99th percentile)

Unlike static VaR models, GARCH-based VaR updates daily with the conditional volatility estimate. Widening during stress periods and tightening during calm regimes.

In [ ]:
z = stats.norm.ppf(0.99)
z

In [ ]:
mean = log_returns.mean()

In [ ]:
VaR_99 = mean - z * cond_vol
VaR_99

In [ ]:
breaches = log_returns < VaR_99
breaches

In [ ]:
breach_days = log_returns[breaches]
breach_days

In [ ]:
len(breach_days)

In [ ]:
len(breach_days) / len(log_returns) * 100

In [ ]:
plt.figure(figsize = (16,8))
plt.plot(log_returns)
plt.plot(VaR_99, color = "red")
plt.scatter(breach_days.index, breach_days.values, color = "Black")
plt.xlabel("Dates", fontsize = 15)
plt.ylabel("Daily Log Returns / VaR", fontsize = 15)
plt.title("ETERNAL.NS — GARCH(1,1) VaR (99%) vs Actual Returns", fontsize = 20)
plt.show()

## 3. Value at Risk (99%)
1-day 99% VaR is computed as:

    VaR = μ - z·σt    
    where z = 2.326 (99th percentile)

Unlike static VaR models, GARCH-based VaR updates daily with the conditional volatility estimate, widening during stress periods and tightening during calm regimes. **Backtesting result: 14 breaches 
out of 1,163 days (1.20%)**, close to the theoretical 1% threshold.

In [ ]:
ES_99 = breach_days.mean()
ES_99

In [ ]:
plt.figure(figsize = (16,8))
plt.plot(log_returns, label = "Log Returns")
plt.plot(VaR_99, color = "red", label = "VaR at 99% CL")
plt.scatter(breach_days.index, breach_days.values, color = "Black", label = "Breach Days")
plt.axhline(ES_99, color = "Green", linestyle = "--", label = "ES at 99% CL")
plt.xlabel("Dates", fontsize = 15)
plt.ylabel("Daily Log Returns / VaR / ES", fontsize = 15)
plt.title("ETERNAL.NS — GARCH(1,1) VaR (99%) vs Actual Returns", fontsize = 20, fontweight='bold')
plt.legend(loc = "best", fontsize = 12)
plt.show()

#### **Prepared by:** Yusuf Sayeed | FRM Part 1 Cleared | FMVA